<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_04_inertia_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 04 — Recovering the Inertia

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: notebooks 00–03.**

The shortest notebook here, and the one most likely to give you a confident
wrong answer.

### Why anyone cares

Inertia used to be a number you looked up. As synchronous plant is replaced by
converter-connected generation, the effective inertia of a region falls, varies
through the day, and is not directly metered. Operators would like to know it
and largely do not.

### Why it is hard

Inertia enters the swing equation through **acceleration**. A system sitting in
steady state contains no information about H at all — the term multiplying it is
zero. You need the grid to be disturbed, which is an uncomfortable experimental
requirement, and it means the window you choose decides whether the answer means
anything.

This is the identifiability lesson from L11.2, in a third domain. The optimiser
will always return a number.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference"); dyn = pb.load("03_dynamic")
V, th, P, Q = ref["V"], ref["th"], ref["P"], ref["Q"]
Y = pb.build_ybus()
machines = pb.Machines()
Y_post = pb.build_ybus(outage=0)
V_p, th_p, _, _ = pb.solve_power_flow(Y_post, P, Q)
Yred_post = pb.reduced_admittance(Y_post, machines, V_p, th_p, P, Q)

# The mechanical powers that generated the trajectory, not the placeholder the
# Machines object is constructed with. Notebook 03's equilibrium() wrote them
# into its own object; identifying H against a different Pm would be fitting
# the wrong model and would push the mismatch straight into H.
machines.Pm = dyn["Pm"]

t_obs, f_obs, delta0 = dyn["t_obs"], dyn["f_obs"], dyn["delta0"]
print(f"true H (the value the simulation used): {machines.H}")
print(f"mechanical power from notebook 03:      {machines.Pm.round(3)}")
print(f"observation window: {t_obs[0]:.2f} to {t_obs[-1]:.2f} s, "
      f"{len(t_obs)} samples")

## 1 · The disturbed window

Fit only the interval where the machines are actually accelerating. Start from a
deliberately wrong initial guess so that recovering the true value means
something — starting at the answer proves nothing.

In [ ]:
mask = (t_obs >= 0.45) & (t_obs <= 2.0)       # around the fault
H_guess = machines.H * np.array([1.0, 2.0])   # machine 1 guessed 2x too large

H_hat, D_hat, net, hist = pb.identify_inertia(
    t_obs[mask], f_obs[mask], machines, Yred_post,
    delta0, np.full(len(machines), machines.ws),
    H_init=H_guess, lam_dyn=1.0, adam_steps=3000, lbfgs_steps=15)

print(f"\n  true  H = {machines.H}")
print(f"  guess H = {H_guess}")
print(f"  found H = {np.round(H_hat, 3)}")
print(f"  error   = {100*abs(H_hat[1]-machines.H[1])/machines.H[1]:.1f}% on machine 1")

fig = pb.plot_loss(hist, "inertia identification"); plt.show()

**Expected output**

> Machine 1's inertia should move substantially from the wrong guess towards the
> true value. Machine 0's will barely move and should not be trusted — it is an
> infinite bus, it does not accelerate, so the data says almost nothing about it.
>
> That asymmetry is the lesson, not a defect. **Report H for machine 1 only**, and
> say why you are not reporting the other.
>
> *(Not executed by the author — see the note in notebook 02.)*

## TODO 1 — the quiet window

Repeat the fit on a window **before** the fault, where nothing is happening.

Record what H comes out as, and how close it stays to your initial guess. Then
answer: how would you know, from the fit alone, that the answer was
meaningless? What diagnostic would you report alongside H?

In [ ]:
# TODO 1 --- the same fit in a quiet window -------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  (t_obs >= 0.0) & (t_obs <= 0.45)          pre-fault only: nothing is accelerating
#   line 2  ->  machines.H * np.array([1.0, 0.5])         a second, different wrong guess
mask_q = ...                                      # <- (t_obs >= 0.0) & (t_obs <= 0.45)
for H_init in (H_guess, ...):                     # <- machines.H * np.array([1.0, 0.5])
    H_q, D_q, _, hist_q = pb.identify_inertia(
        t_obs[mask_q], f_obs[mask_q], machines, Yred_post,
        delta0, np.full(len(machines), machines.ws),
        H_init=H_init, lam_dyn=1.0, adam_steps=3000, lbfgs_steps=15, verbose=False)
    print(f"  guess H = {np.round(H_init, 3)}  ->  found H = {np.round(H_q, 3)}"
          f"   data {hist_q['data'][-1]:.2e}   dyn {hist_q['dyn'][-1]:.2e}")
print(f"\n  disturbed window gave  data {hist['data'][-1]:.2e}   dyn {hist['dyn'][-1]:.2e}")
print("  The loss is no worse in the quiet window -- and the answer depends on the guess. That is the point.")
# ------------------------------------------------------------------------------

## TODO 2 — inertia from RoCoF, without any network

There is a much simpler estimate available. For a power imbalance ΔP, the
initial rate of change of frequency is set by the stored kinetic energy:

$$\mathrm{RoCoF}_0 = \frac{f_0\,\Delta P}{2E_k},
\qquad E_k = \sum_i H_i S_{n,i}$$

So a measured RoCoF and a known trip size give you E_k directly. Try it on the
frequency event from `problem.py`, and compare with your fitted H.

In [ ]:
event = pb.nordic_frequency()
for window in (0.5, 1.0, 2.0):
    r, t_ev = pb.rocof_from_series(event["t"], event["f_hz"], window=window)
    print(f"  window {window:3.1f} s ->  RoCoF {r:+.3f} Hz/s")
print(f"\n  the event was built with a true initial RoCoF of "
      f"{event['true_rocof']:+.2f} Hz/s")
fig = pb.plot_frequency_event(
    event["t"], event["f_hz"], t_ev, 1.0,
    fit=pb.rocof_from_series(event["t"], event["f_hz"], 1.0)[0])
plt.show()

# TODO 2 --- invert the RoCoF relation for the kinetic energy ----------------------------------------------
# Two `...` to replace:
#   line 1  ->  1000.0                                the trip size you assume, MW (state it in the report)
#   line 2  ->  50.0 * DP_MW / (2.0 * abs(r))         E_k = f0 dP / (2 RoCoF): the inverse of pb.rocof_initial
DP_MW = ...                                       # <- 1000.0
print(f"\n  kinetic energy of the study network: {pb.kinetic_energy(machines):,.0f} MWs")
for window in (0.5, 1.0, 2.0):
    r, _ = pb.rocof_from_series(event["t"], event["f_hz"], window=window)
    Ek = ...                                      # <- 50.0 * DP_MW / (2.0 * abs(r))
    print(f"  window {window:3.1f} s:  RoCoF {r:+.3f} Hz/s  ->  E_k {Ek:,.0f} MWs  for a {DP_MW:.0f} MW trip")
# Three windows, three answers. Which is right, and what does "right" mean here?
# ------------------------------------------------------------------------------

**Expected output**

> Three windows, three different RoCoF values — roughly **-0.21, -0.19 and -0.14
> Hz/s** for 0.5, 1.0 and 2.0 second windows, against a true initial slope of
> **-0.28 Hz/s**.
>
> This is not a bug. The frequency decline is curved, so a longer window averages
> in the flattening and biases the estimate towards zero. Shorter windows are
> closer to the true initial slope but noisier.
>
> **This is why RoCoF measurement windows are standardised**, and why a RoCoF
> quoted without its window is not a number you can use.

In [ ]:
pb.save("04_inertia", H_hat=H_hat, D_hat=D_hat, H_true=machines.H)
print("\nnotebook 04 complete — go to 05_compare_and_report")